In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.data import load_clean, filter_modeling, time_split
from src.features import (add_features, build_structured_transformer,
                          build_name_vectorizer, transform_features)
from src.modeling import evaluate, segment_metrics

REP = ROOT / "reports"; (REP / "figures").mkdir(parents=True, exist_ok=True)

df = add_features(filter_modeling(load_clean(ROOT / "SBAnational.csv")))
parts = time_split(df)
for k, v in parts.items():
    print(k, v.shape, "부실률:", round(v["target"].mean(), 4))

st = build_structured_transformer().fit(parts["train"])
nv = build_name_vectorizer().fit(parts["train"]["Name"].astype("string").fillna(""))

train (727785, 32) 부실률: 0.1414
valid (71639, 32) 부실률: 0.4279
test (75296, 32) 부실률: 0.2991


In [2]:
def make_xy(part, include_name):
    X = transform_features(st, nv, part, include_name=include_name)
    y = part["target"].astype(int).values
    return X, y

Xtr_s, ytr = make_xy(parts["train"], False)
Xte_s, yte = make_xy(parts["test"], False)
Xtr_sn, _ = make_xy(parts["train"], True)
Xte_sn, _ = make_xy(parts["test"], True)
is_new_test = parts["test"]["is_new"].values
print("정형:", Xtr_s.shape, "| 정형+Name:", Xtr_sn.shape)

정형: (727785, 153) | 정형+Name: (727785, 5153)


In [3]:
from sklearn.linear_model import LogisticRegression

results = {}
def fit_eval_logreg(Xtr, Xte, tag):
    clf = LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=-1)
    clf.fit(Xtr, ytr)
    s = clf.predict_proba(Xte)[:, 1]
    results[tag] = evaluate(yte, s)
    print(tag, results[tag])
    return clf, s

logreg_s, s_logreg_s = fit_eval_logreg(Xtr_s, Xte_s, "logreg | 정형")
logreg_sn, s_logreg_sn = fit_eval_logreg(Xtr_sn, Xte_sn, "logreg | 정형+Name")

logreg | 정형 {'pr_auc': 0.5032232570528725, 'roc_auc': 0.750246536185399, 'precision': 0.42694068977722716, 'recall': 0.8568954799751354, 'f1': 0.569922480620155}


logreg | 정형+Name {'pr_auc': 0.5149791880179692, 'roc_auc': 0.7550183377920189, 'precision': 0.43465234726095275, 'recall': 0.837403427759524, 'f1': 0.572269506774081}


In [4]:
import lightgbm as lgb

spw = (ytr == 0).sum() / (ytr == 1).sum()
print("scale_pos_weight:", round(spw, 3))

def fit_eval_lgbm(Xtr, Xte, tag):
    clf = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                             subsample=0.8, colsample_bytree=0.8,
                             scale_pos_weight=spw, n_jobs=-1, random_state=0)
    clf.fit(Xtr, ytr)
    s = clf.predict_proba(Xte)[:, 1]
    results[tag] = evaluate(yte, s)
    print(tag, results[tag])
    return clf, s

lgbm_s, s_lgbm_s = fit_eval_lgbm(Xtr_s, Xte_s, "lgbm | 정형")
lgbm_sn, s_lgbm_sn = fit_eval_lgbm(Xtr_sn, Xte_sn, "lgbm | 정형+Name")

scale_pos_weight: 6.074


[LightGBM] [Info] Number of positive: 102883, number of negative: 624902
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.062988 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1830
[LightGBM] [Info] Number of data points in the train set: 727785, number of used features: 151


[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.141365 -> initscore=-1.804002
[LightGBM] [Info] Start training from score -1.804002


C:\Users\hany1\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


lgbm | 정형 {'pr_auc': 0.8881978573759661, 'roc_auc': 0.9466862274627669, 'precision': 0.7473245257952617, 'recall': 0.8991652606340467, 'f1': 0.8162434502216848}


...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 17.619228 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1026023


[LightGBM] [Info] Number of data points in the train set: 727785, number of used features: 5151


[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.141365 -> initscore=-1.804002
[LightGBM] [Info] Start training from score -1.804002


C:\Users\hany1\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


lgbm | 정형+Name {'pr_auc': 0.8895752302767406, 'roc_auc': 0.9496462804312927, 'precision': 0.7432214568371684, 'recall': 0.9006304946274754, 'f1': 0.8143895290480587}
